### LOADING CSV DATA

In [1]:
import pandas as pd

orders = pd.read_csv("../data/olist_orders_dataset.csv")

display(orders.head())
print(f"Rows: {orders.shape[0]}, Columns: {orders.shape[1]}")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Rows: 99441, Columns: 8


### Data type and Missing values:

In [2]:
# Data type of each column
display(orders.dtypes)
# Missing values in each column
display(orders.isna().sum())


order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

### DUPLICATES:

In [3]:
# Duplicate complete rows
print(orders.duplicated().sum())

# Duplicate order_id values
print(orders["order_id"].duplicated().sum())

0
0


### COUNT using VALUE_COUNTS

In [4]:
print(orders["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


### CONVERSION "TO_DATE_TIME"

In [5]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)
print(orders["order_purchase_timestamp"].dtype)
##---------------------------------------------------------
orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"]
)
print(orders["order_approved_at"].dtype)
##---------------------------------------------------------
orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"]
)
print(orders["order_delivered_carrier_date"].dtype)
##---------------------------------------------------------
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)
print(orders["order_delivered_customer_date"].dtype)
##---------------------------------------------------------
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)
print(orders["order_estimated_delivery_date"].dtype)

display(orders.dtypes)



datetime64[us]
datetime64[us]
datetime64[us]
datetime64[us]
datetime64[us]


order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [6]:
delivered_orders = orders.loc[orders["order_status"] == "delivered"].copy()

delivered_orders[["order_purchase_timestamp", "order_delivered_customer_date"]].isna().sum()

order_purchase_timestamp         0
order_delivered_customer_date    8
dtype: int64

In [7]:
valid_delivery_orders = delivered_orders.loc[
    delivered_orders["order_purchase_timestamp"].notna()
    & delivered_orders["order_delivered_customer_date"].notna()
    & (
        delivered_orders["order_delivered_customer_date"]
        >= delivered_orders["order_purchase_timestamp"]
    )
].copy()


In [8]:
valid_delivery_orders["delivery_days"] = (
    valid_delivery_orders["order_delivered_customer_date"]
    - valid_delivery_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

average_delivery_days = round(valid_delivery_orders["delivery_days"].mean(), 2)
print(f"Average delivery days: {average_delivery_days:.2f}")

Average delivery days: 12.56


#### Classify deliveries as on time or late.

Using valid_delivery_orders:

1. Create delivery_performance, keeping only rows with a non-missing order_estimated_delivery_date.
2. Add a column called delivery_group:
"On time" when the actual delivery date is on or before the estimated date.
"Late" when the actual delivery date is after the estimated date.

Compare calendar dates only, ignoring the time of day

In [9]:
delivery_performance = valid_delivery_orders.loc[
    valid_delivery_orders["order_estimated_delivery_date"].notna()
].copy()

# Compare calendar dates only, ignoring the time of day.
on_time = (
    delivery_performance["order_delivered_customer_date"].dt.normalize()
    <= delivery_performance["order_estimated_delivery_date"].dt.normalize()
)
delivery_performance["delivery_group"] = on_time.map(
    {True: "On time", False: "Late"}
)

Calculate on-time and late delivery percentages.
Using delivery_performance, 

create a summary with:

delivery_group — "On time" or "Late"
order_count — number of orders in each group
percentage — each group’s percentage of all orders in delivery_performance, rounded to 2 decimal places

In [10]:
order_counts = delivery_performance["delivery_group"].value_counts()
total_orders = len(delivery_performance)

percentages = (order_counts / total_orders * 100).round(2)
print(f"Percentage of on-time deliveries: {percentages['On time']:.2f}%")

delivery_summary = pd.DataFrame({
    "order_count": order_counts,
    "percentage" : percentages 
}) 

delivery_summary = delivery_summary.rename_axis("delivery_group").reset_index()

Percentage of on-time deliveries: 93.23%


In [11]:
delivery_summary.head()

,delivery_group,order_count,percentage
0,On time,89936,93.23
1,Late,6534,6.77


Load and check the customer data.

1.Load ../data/olist_customers_dataset.csv into a DataFrame named customers.

2.Check its number of rows and columns.

3.Count missing values in customer_id and customer_unique_id.

4.Count duplicate values in customer_id.

We’re checking customer_id because we’ll use it to join customers with orders.

In [12]:
import pandas as pd

customers = pd.read_csv("../data/olist_customers_dataset.csv")

display(customers.head())
print(f"Rows: {customers.shape[0]}, Columns: {customers.shape[1]}")
display(customers[["customer_id", "customer_unique_id"]].isna().sum())


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Rows: 99441, Columns: 5


customer_id           0
customer_unique_id    0
dtype: int64

Is delivery performance by state. We’ll start by joining the tables:

In [13]:
delivery_with_customers = delivery_performance.merge(
    customers[["customer_id", "customer_state"]],
    on = "customer_id",
    how = "left",
    validate="many_to_one"
)
delivery_with_customers.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_group,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,On time,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,On time,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,On time,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,On time,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,On time,SP


Count total orders and late orders for each state and the final step of question 4 is to rank states by late-delivery percentage.

In [14]:
delivery_with_customers["is_late"] = (
    delivery_with_customers["delivery_group"] == "Late"
)
state_summary = delivery_with_customers.groupby(
    "customer_state", dropna=False
).agg(
    orders_analyzed=("order_id", "size"),
    late_orders=("is_late", "sum")
).reset_index()

state_summary["late_percentage"] = (
    state_summary["late_orders"] / state_summary["orders_analyzed"] * 100
).round(2)

An order can have multiple reviews. Calculate its average score first so that each order gets equal weight. Exclude orders without a usable review score from this comparison.

In [15]:
import pandas as pd

# 1. Load the reviews
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

# 2. Calculate one average review score per order
order_avg_reviews = (
    reviews.loc[reviews["review_score"].notna()]
    .groupby("order_id")["review_score"]
    .mean()
    .reset_index(name="order_avg_score")
)

# 3. Keep delivery orders with usable review scores
delivery_reviews = delivery_performance[
    ["order_id", "delivery_group"]
].merge(
    order_avg_reviews,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

# 4. Summarize reviews by delivery group
delivery_review_summary = (
    delivery_reviews.groupby("delivery_group")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        avg_review_score=("order_avg_score", "mean")
    )
    .reset_index()
)

# 5. Round the average scores
delivery_review_summary["avg_review_score"] = (
    delivery_review_summary["avg_review_score"].round(2)
)

# 6. Show "On time" before "Late"
delivery_review_summary = (
    delivery_review_summary
    .sort_values("delivery_group", ascending=False)
    .reset_index(drop=True)
)

display(delivery_review_summary)

,delivery_group,reviewed_orders,avg_review_score
0,On time,89443,4.29
1,Late,6381,2.27


Calculate the average total payment per order.

In [16]:
import pandas as pd

payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
order_payment_totals = (
    payments.loc[payments["payment_value"].notna()]
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index(name="total_payment_value")
)
average_payment_value = round(order_payment_totals["total_payment_value"].mean(), 2)
payment_average_difference = round(average_payment_value - 160.99, 2)
print(f"Difference from SQL average payment per order: {payment_average_difference:.2f}")



Difference from SQL average payment per order: 0.00


In [17]:
delivered_customers_orders = delivered_orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

customer_order_counts = (
    delivered_customers_orders.groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index(name="total_orders")
)

repeat_customer_percentage = round(
    (customer_order_counts["total_orders"] > 1).sum()
    / len(customer_order_counts) * 100,
    2
)

Find the top 10 product categories by merchandise sales from delivered orders. Use price, excluding freight.

In [18]:
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")

items_with_categories = order_items.merge(
    products[["product_id", "product_category_name"]],
    on = "product_id",
    how="left",
    validate="many_to_one"
)    

delivered_items = items_with_categories.merge(
    delivered_orders[["order_id"]],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

Top 10 categories by delivered merchandise sales using delivered_items.

In [19]:
top_categories = (
    delivered_items.groupby("product_category_name")["price"].sum()
    .reset_index(name="total_merchandise_sales_value")
    .sort_values("total_merchandise_sales_value", ascending=False)
    .head(10)
)

1. Calculate monthly payments and month-over-month growth
2. calculate total payments for each purchase month
3. Calculate the month-over-month payment growth percentage.

In [20]:
import numpy as np
order_with_payments = order_payment_totals.merge(
    orders[["order_id","order_purchase_timestamp"]],
    on="order_id",
    how="inner",
    validate="one_to_one"
)

monthly_payments = order_with_payments.loc[
    order_with_payments["order_purchase_timestamp"].notna()
].copy()
monthly_payments["purchase_month"] = monthly_payments["order_purchase_timestamp"].dt.to_period("M")
monthly_payments = monthly_payments.groupby("purchase_month")
monthly_payment_totals = monthly_payments["total_payment_value"].sum().sort_index()
all_months = pd.period_range(
    start=monthly_payment_totals.index.min(),
    end=monthly_payment_totals.index.max(),
    freq="M",
)
monthly_payment_totals = monthly_payment_totals.reindex(all_months, fill_value=0)
sorted_monthly_payments = monthly_payment_totals.rename_axis("purchase_month").reset_index(
    name="monthly_payment_total"
)

previous_month_payment = sorted_monthly_payments.copy()
previous_month_payment["previous_month_payment"] = previous_month_payment[
    "monthly_payment_total"
].shift(1)
monthly_growth_percentage = previous_month_payment.copy()
monthly_growth_percentage["month_over_month_growth"] = (
    monthly_growth_percentage["monthly_payment_total"]
    - monthly_growth_percentage["previous_month_payment"]
) / monthly_growth_percentage["previous_month_payment"] * 100

monthly_growth_percentage["month_over_month_growth"] = (
    monthly_growth_percentage["month_over_month_growth"].round(2)
)
monthly_growth_percentage.loc[
    monthly_growth_percentage["previous_month_payment"] == 0,
    "month_over_month_growth"
] = np.nan
monthly_growth_percentage.head()

,purchase_month,monthly_payment_total,previous_month_payment,month_over_month_growth
0,2016-09,252.24,NaN,NaN
1,2016-10,59090.48,252.24,23326.29
2,2016-11,0.00,59090.48,-100.00
3,2016-12,19.62,0.00,NaN
4,2017-01,138488.04,19.62,705751.38


Which product categories generate high revenue but have relatively low review scores?

In [21]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. MONTHLY PAYMENTS AND MONTH-OVER-MONTH GROWTH
# ============================================================

# Convert purchase dates again because reloading orders resets
# CSV date columns to text.
order_dates = orders[["order_id", "order_purchase_timestamp"]].copy()

order_dates["order_purchase_timestamp"] = pd.to_datetime(
    order_dates["order_purchase_timestamp"]
)

orders_with_payments = order_payment_totals.merge(
    order_dates,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

orders_with_payments = orders_with_payments.loc[
    orders_with_payments["order_purchase_timestamp"].notna()
].copy()

orders_with_payments["purchase_month"] = (
    orders_with_payments["order_purchase_timestamp"].dt.to_period("M")
)

monthly_payments = (
    orders_with_payments
    .groupby("purchase_month")["total_payment_value"]
    .sum()
    .reset_index()
    .sort_values("purchase_month")
    .reset_index(drop=True)
)

# Check that the preceding row is the previous calendar month.
previous_month = monthly_payments["purchase_month"].shift(1)

consecutive_months = (
    previous_month == monthly_payments["purchase_month"] - 1
)

monthly_payments["previous_month_payment"] = (
    monthly_payments["total_payment_value"]
    .shift(1)
    .where(consecutive_months)
)

# Leave growth missing when the previous payment is missing or zero.
previous_payment = monthly_payments["previous_month_payment"]

safe_denominator = previous_payment.where(previous_payment != 0)

monthly_payments["mom_growth_percentage"] = (
    (
        monthly_payments["total_payment_value"] - previous_payment
    )
    / safe_denominator * 100
).round(2)

monthly_payments[
    ["total_payment_value", "previous_month_payment"]
] = monthly_payments[
    ["total_payment_value", "previous_month_payment"]
].round(2)

display(monthly_payments)


# ============================================================
# 2. COMPARE KEY PYTHON RESULTS WITH YOUR EARLIER SQL OUTPUTS
# ============================================================

review_check = delivery_review_summary.set_index("delivery_group")

validation_report = pd.DataFrame(
    [
        (
            "Average delivery days",
            valid_delivery_orders["delivery_days"].mean(),
            12.56
        ),
        (
            "Average payment per order",
            order_payment_totals["total_payment_value"].mean(),
            160.99
        ),
        (
            "Customers with delivered orders",
            len(customer_order_counts),
            93358
        ),
        (
            "Repeat customers",
            (customer_order_counts["total_orders"] > 1).sum(),
            2801
        ),
        (
            "Repeat customer percentage",
            (customer_order_counts["total_orders"] > 1).mean() * 100,
            3.00
        ),
        (
            "Orders analyzed for delivery performance",
            len(delivery_performance),
            96470
        ),
        (
            "On-time orders",
            (delivery_performance["delivery_group"] == "On time").sum(),
            89936
        ),
        (
            "Late orders",
            (delivery_performance["delivery_group"] == "Late").sum(),
            6534
        ),
        (
            "Reviewed on-time orders",
            review_check.loc["On time", "reviewed_orders"],
            89443
        ),
        (
            "Reviewed late orders",
            review_check.loc["Late", "reviewed_orders"],
            6381
        ),
        (
            "On-time average review score",
            review_check.loc["On time", "avg_review_score"],
            4.29
        ),
        (
            "Late average review score",
            review_check.loc["Late", "avg_review_score"],
            2.27
        )
    ],
    columns=["metric", "python_value", "sql_value"]
)

validation_report["python_value"] = (
    validation_report["python_value"].round(2)
)

validation_report["difference"] = (
    validation_report["python_value"] - validation_report["sql_value"]
).round(2)

validation_report["status"] = (
    validation_report["difference"].eq(0)
    .map({True: "PASS", False: "CHECK"})
)

# Check that previous-month values are missing across calendar gaps.
gap_check_passed = monthly_payments.loc[
    ~consecutive_months,
    ["previous_month_payment", "mom_growth_percentage"]
].isna().all().all()

display(validation_report)
print("Missing-month handling:", "PASS" if gap_check_passed else "CHECK")


# ============================================================
# 3. EXPORT THE COMPLETED TABLES
# ============================================================

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Store month labels as YYYY-MM in the exported CSV.
monthly_export = monthly_payments.copy()
monthly_export["purchase_month"] = (
    monthly_export["purchase_month"].astype(str)
)

tables_to_export = {
    "delivery_summary.csv": delivery_summary,
    "state_delivery_summary.csv": state_summary,
    "delivery_review_summary.csv": delivery_review_summary,
    "order_payment_totals.csv": order_payment_totals,
    "customer_order_counts.csv": customer_order_counts,
    "top_categories.csv": top_categories,
    "monthly_payments.csv": monthly_export,
    "validation_report.csv": validation_report
}

for filename, table in tables_to_export.items():
    table.to_csv(output_dir / filename, index=False)

print(f"\nSaved {len(tables_to_export)} CSV files to:")
print(output_dir.resolve())

if validation_report["status"].eq("PASS").all() and gap_check_passed:
    print("\nAll listed validation checks passed.")
else:
    print("\nSome checks need review. See validation_report above.")

,purchase_month,total_payment_value,previous_month_payment,mom_growth_percentage
0,2016-09,252.24,NaN,NaN
1,2016-10,59090.48,252.24,23326.29
2,2016-12,19.62,NaN,NaN
3,2017-01,138488.04,19.62,705751.38
4,2017-02,291908.01,138488.04,110.78
5,2017-03,449863.60,291908.01,54.11
6,2017-04,417788.03,449863.60,-7.13
7,2017-05,592918.82,417788.03,41.92
8,2017-06,511276.38,592918.82,-13.77
9,2017-07,592382.92,511276.38,15.86


,metric,python_value,sql_value,difference,status
0,Average delivery days,12.56,12.56,0.0,PASS
1,Average payment per order,160.99,160.99,0.0,PASS
2,Customers with delivered orders,93358.00,93358.00,0.0,PASS
3,Repeat customers,2801.00,2801.00,0.0,PASS
4,Repeat customer percentage,3.00,3.00,0.0,PASS
5,Orders analyzed for delivery performance,96470.00,96470.00,0.0,PASS
6,On-time orders,89936.00,89936.00,0.0,PASS
7,Late orders,6534.00,6534.00,0.0,PASS
8,Reviewed on-time orders,89443.00,89443.00,0.0,PASS
9,Reviewed late orders,6381.00,6381.00,0.0,PASS


Missing-month handling: PASS

Saved 8 CSV files to:
C:\Users\DEEPAK\OneDrive\Desktop\Ecommerce_Sales_Customer_Analytics\outputs

All listed validation checks passed.


In [22]:
%run ecommerce_analysis_complete.py

Loading CSVs from: C:\Users\DEEPAK\OneDrive\Desktop\Ecommerce_Sales_Customer_Analytics\data


C:\Users\DEEPAK\OneDrive\Desktop\Ecommerce_Sales_Customer_Analytics\python\ecommerce_analysis_complete.py:233: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  monthly_payments = monthly_payments.round(2)



SQL benchmark checks:
                    metric  python_value  sql_value  difference status
     Average delivery days         12.56      12.56         0.0   PASS
 Average payment per order        160.99     160.99         0.0   PASS
       Delivered customers      93358.00   93358.00         0.0   PASS
          Repeat customers       2801.00    2801.00         0.0   PASS
Repeat customer percentage          3.00       3.00         0.0   PASS
  Delivery orders analyzed      96470.00   96470.00         0.0   PASS
            On-time orders      89936.00   89936.00         0.0   PASS
               Late orders       6534.00    6534.00         0.0   PASS
   Reviewed on-time orders      89443.00   89443.00         0.0   PASS
      Reviewed late orders       6381.00    6381.00         0.0   PASS
    On-time average review          4.29       4.29         0.0   PASS
       Late average review          2.27       2.27         0.0   PASS

Priority categories (top 10 named categories by sales